# Analiza wypadków drogowych w USA — EDA i przygotowanie danych do modelowania (ZSSI)

**Zakres notatnika (Etapy 1–4 projektu):**
1. **Określenie zadania i środowiska:** wieloklasowa klasyfikacja stopnia ciężkości wypadku (`Severity`, klasy 1–4) w momencie jego wystąpienia ($t_0$) na podstawie zbioru *US Accidents (March 2023)*. Technologia: Python (`pandas`, `numpy`, `matplotlib`, `seaborn`, `missingno`, `scikit-learn`, `imbalanced-learn`).
2. **Eksploracyjna analiza danych (EDA):** klasyfikacja atrybutów, statystyki opisowe z modą, audyt duplikatów bez `ID`, braki na poziomie atrybutów, obserwacji oraz per klasa, synonimy kategorii, współczynnik niezbalansowania $IR$, pokrycie kalendarza, rozkłady w podziale na klasy, korelacja rang Spearmana, błędy czujników vs filtr IQR oraz wnioski pod dalsze przetwarzanie.
3. **Przygotowanie zbioru danych do utworzenia modelu:**
   - deduplikacja zdarzeń bez `ID` i ścisły podział Hold-out (80/20) na surowych cechach (z wykluczeniem cech wyciekowych `Source`, `Distance(mi)`, `End_Time`),
   - własny transformator **`AccidentFeatureEngineer(BaseEstimator, TransformerMixin)`** zgodny z wytycznymi *scikit-learn developers* (`fit`/`transform`, `clone`, `set_params`, brak mutacji stanu w `transform`),
   - kompletny, bezwyciekowy **`Pipeline` / `ImbPipeline`** łączący inżynierię cech, `ColumnTransformer`, `VarianceThreshold` oraz strategie balansowania klas,
   - 5-krotna warstwowa walidacja krzyżowa (`StratifiedKFold`) oraz gotowa fabryka **`GridSearchCV`** z siatką hiperparametrów preprocessingu (`step__param`) i metrykami pod niezbalansowane dane.

*Notatnik nie trenuje ani nie ocenia modeli uczenia maszynowego (etapy 5–9 realizowane są w kolejnej fazie projektu).*

## 1. Konfiguracja środowiska i wczytanie danych

Import bibliotek, definicja stałych projektowych oraz wczytanie `US_Accidents_March23.csv` z optymalizacją typów danych (`category`, `float32`, `int8`), ograniczającą zużycie pamięci RAM do ok. 1 GB.

In [ ]:
import gc
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import missingno as mn
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, f1_score, make_scorer, matthews_corrcoef, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.utils.validation import check_is_fitted

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

LOCAL_CSV_PATH = "US_Accidents_March23.csv"
PREPARED_ARTIFACT_PATH = "prepared_data_stage4.joblib"
TARGET_COLUMN = "Severity"

SAMPLE_LIMIT = 100_000  # Próbka warstwowa przygotowywana pod późniejsze modelowanie
TEST_SIZE = 0.20        # Zamrożony zbiór testowy (Hold-out)
CV_FOLDS = 5            # Liczba foldów walidacji krzyżowej
RANDOM_STATE = 42
N_JOBS = 1              # 1 wątek zapobiega błędowi pamięci WinError 1455 w Windows

# Sztywne granice fizyczne odczytów czujników pogodowych (reguły bezstanowe -> brak wycieku danych)
PHYSICAL_BOUNDS = {
    "Temperature(F)": (-60.0, 135.0),
    "Wind_Chill(F)": (-80.0, 135.0),
    "Humidity(%)": (0.0, 100.0),
    "Pressure(in)": (25.0, 32.0),
    "Visibility(mi)": (0.0, 50.0),
    "Wind_Speed(mph)": (0.0, 150.0),
    "Precipitation(in)": (0.0, 20.0),
}

# Ujednolicenie równoważnych zapisów kierunku wiatru
WIND_ALIASES = {
    "CALM": "Calm",
    "VAR": "Variable",
    "East": "E",
    "West": "W",
    "North": "N",
    "South": "S",
}

# Klucz identyfikacji duplikatów zdarzeń z pominięciem sztucznego ID
DUP_SUBSET = ["Start_Time", "End_Time", "Start_Lat", "Start_Lng", "Severity", "Street", "City", "State"]

DTYPE_MAP = {
    **{c: "category" for c in [
        "Source", "State", "Country", "Timezone", "Airport_Code",
        "Weather_Condition", "Wind_Direction", "Sunrise_Sunset",
        "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",
    ]},
    **{c: "float32" for c in [
        "Start_Lat", "Start_Lng", "End_Lat", "End_Lng", "Distance(mi)",
        "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
        "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",
    ]},
    "Severity": "int8",
}

if os.path.exists(LOCAL_CSV_PATH):
    df = pd.read_csv(LOCAL_CSV_PATH, dtype=DTYPE_MAP, low_memory=False)
else:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "sobhanmoosavi/us-accidents",
        LOCAL_CSV_PATH,
        pandas_kwargs={"dtype": DTYPE_MAP, "low_memory": False},
    )

print(f"Wczytano zbiór: {df.shape[0]:,} wierszy × {df.shape[1]} kolumn")
print(f"Zużycie pamięci: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
df.head()

## 2. Eksploracyjna analiza danych (EDA)

### 2.1 Klasyfikacja atrybutów, podstawowe statystyki (z modą) oraz audyt jakości danych

Podział atrybutów na jakościowe i ilościowe, wyznaczenie statystyk opisowych wraz z modą, wykrycie cech stałych i synonimów w `Wind_Direction`, kontrola duplikatów z pominięciem `ID` oraz analiza braków na poziomie atrybutów, obserwacji i klas `Severity`.

In [ ]:
# 1. Klasyfikacja typów atrybutów
attribute_classification = pd.DataFrame([
    {"Typ główny": "Jakościowy", "Podtyp": "Porządkowy (ordinal)", "Liczba": 1,
     "Kolumny": "Severity (zmienna celu: 1 < 2 < 3 < 4)"},
    {"Typ główny": "Jakościowy", "Podtyp": "Nominalny (nominal)", "Liczba": 15,
     "Kolumny": "Source, Street, City, County, State, Zipcode, Country, Timezone, Airport_Code, Wind_Direction, Weather_Condition, Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight"},
    {"Typ główny": "Ilościowy", "Podtyp": "Dyskretny / binarny (discrete)", "Liczba": 13,
     "Kolumny": "Amenity, Bump, Crossing, Give_Way, Junction, No_Exit, Railway, Roundabout, Station, Stop, Traffic_Calming, Traffic_Signal, Turning_Loop"},
    {"Typ główny": "Ilościowy", "Podtyp": "Ciągły (continuous)", "Liczba": 12,
     "Kolumny": "Start_Lat, Start_Lng, End_Lat, End_Lng, Distance(mi), Temperature(F), Wind_Chill(F), Humidity(%), Pressure(in), Visibility(mi), Wind_Speed(mph), Precipitation(in)"},
    {"Typ główny": "Techniczny / tekst", "Podtyp": "Identyfikator, tekst, czas", "Liczba": 5,
     "Kolumny": "ID, Description, Start_Time, End_Time, Weather_Timestamp"},
])
display(attribute_classification)

df.info()

# 2. Statystyki opisowe kolumn numerycznych (z modą) oraz kategorycznych
num_summary = df.describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
num_summary["mode"] = df[num_summary.index].mode(dropna=True).iloc[0]
display(num_summary.round(3))

cat_summary = df.describe(include="category").T
cat_summary["missing_%"] = (df[cat_summary.index].isna().mean() * 100).round(3)
display(cat_summary)

# 3. Cechy stałe i synonimy w Wind_Direction
zero_var_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print("Kolumny stałe (zerowa wariancja):", zero_var_cols)

raw_wind = df["Wind_Direction"].dropna().unique().astype(str)
wind_map_df = pd.DataFrame({"oryginał": raw_wind, "ujednolicona": [WIND_ALIASES.get(w, w) for w in raw_wind]})
print(f"Wind_Direction: {len(raw_wind)} kategorii surowych -> {wind_map_df['ujednolicona'].nunique()} po ujednoliceniu")
display(wind_map_df[wind_map_df["oryginał"] != wind_map_df["ujednolicona"]])

# 4. Wykrywanie duplikatów (pułapka unikalnego ID)
id_dups = int(df["ID"].duplicated().sum()) if "ID" in df.columns else 0
dup_cols = [c for c in DUP_SUBSET if c in df.columns]
event_dups = int(df.duplicated(subset=dup_cols).sum())
print(f"Duplikaty po ID: {id_dups:,}")
print(f"Duplikaty zdarzeń bez ID (po {dup_cols}): {event_dups:,} ({event_dups / len(df) * 100:.3f}%)")

# 5. Braki na poziomie atrybutów, klas Severity oraz obserwacji (wierszy)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
display(missing_pct.to_frame(name="missing_%").round(3))

weather_audit_cols = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)", "Weather_Condition",
]
missing_by_class = df.groupby(TARGET_COLUMN, observed=True)[weather_audit_cols].apply(
    lambda g: g.isna().mean() * 100
)
print("Braki danych pogodowych w podziale na klasy Severity (%):")
display(missing_by_class.round(2))

row_missing_counts = df.isna().sum(axis=1).value_counts().sort_index()
row_missing_df = pd.DataFrame({
    "liczba_wierszy": row_missing_counts,
    "udział_%": (row_missing_counts / len(df) * 100).round(3),
})
row_missing_df.index.name = "liczba_braków_w_wierszu"
display(row_missing_df.head(12))

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=row_missing_counts.index.astype(str), y=row_missing_counts.values, hue=row_missing_counts.index.astype(str), palette="crest", dodge=False, ax=ax)
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set_title("Rozkład liczby brakujących wartości na obserwację (wiersz)")
ax.set_xlabel("Liczba braków w wierszu")
ax.set_ylabel("Liczba obserwacji")
plt.tight_layout()
plt.show()

mn.matrix(df[missing_pct.index].sample(n=min(50_000, len(df)), random_state=RANDOM_STATE))
plt.title("Wzorzec brakujących wartości (próbka 50 tys. rekordów)")
plt.show()

**Wnioski (2.1):**
- Zmienna docelowa `Severity` jest atrybutem **jakościowym porządkowym** (4 poziomy), kolumny lokalizacyjno-pogodowe są nominalne, odczyty czujników ciągłe, a flagi infrastruktury dyskretne/binarne.
- Kolumny `Country` i `Turning_Loop` mają zerową wariancję, a `End_Lat`/`End_Lng` ponad 44% braków — należy je odrzucić. `Wind_Direction` zawiera synonimy (`CALM`/`Calm`, `VAR`/`Variable`, `West`/`W` itp.) wymagające ujednolicenia.
- Mimo unikalności `ID` występują zduplikowane zdarzenia bez `ID` — należy je usunąć przed podziałem danych, aby zapobiec przeciekowi między zbiorem treningowym a walidacyjnym/testowym.
- Braki pogodowe **nie są losowe** (silne zróżnicowanie odsetka braków `Wind_Chill(F)` i `Precipitation(in)` między klasami `Severity`), dlatego w etapie przygotowania należy dodać jawne flagi braków zamiast usuwać wiersze.

### 2.2 Rozkład zmiennej docelowej (`Severity`) i współczynnik niezbalansowania $IR$

Wyznaczenie udziałów procentowych klas oraz współczynnika niezbalansowania $IR = \frac{N^-}{N^+}$ ($N^-$ — liczebność dominującej klasy większościowej, $N^+$ — liczebność klasy mniejszościowej).

In [ ]:
sev_counts = df[TARGET_COLUMN].value_counts().sort_index()
sev_share = (sev_counts / len(df) * 100).round(2)
n_majority = int(sev_counts.max())
maj_class = int(sev_counts.idxmax())
ir_per_class = (n_majority / sev_counts).round(2)
ir_max = n_majority / int(sev_counts.min())

sev_table = pd.DataFrame({
    "liczba": sev_counts,
    "udział_%": sev_share,
    f"IR (N_{maj_class} / N_c)": ir_per_class,
})
display(sev_table)
print(f"Maksymalny współczynnik niezbalansowania IR_max (klasa {maj_class} vs {int(sev_counts.idxmin())}): {ir_max:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=sev_counts.index.astype(str), y=sev_counts.values, hue=sev_counts.index.astype(str), palette="crest_r", dodge=False, ax=ax)
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set_title("Rozkład stopni ciężkości wypadków (Severity) — wykres słupkowy")
ax.set_xlabel("Severity (atrybut porządkowy)")
ax.set_ylabel("Liczba wypadków")
for patch, pct, ir_val in zip(ax.patches, sev_share.values, ir_per_class.values):
    ax.annotate(f"{pct:.2f}%\n(IR={ir_val:.1f})", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

**Wnioski (2.2):**
Zbiór jest skrajnie niezbalansowany: klasa `2` stanowi ~79,7% rekordów, podczas gdy klasy `1` (~0,9%) i `4` (~2,6%) są silnie mniejszościowe ($IR_{\max} \approx 91$, dla klasy `4` $IR \approx 30$). Wymaga to warstwowego podziału (`stratify`), mechanizmów balansowania klas w Etapie 4 oraz unikania zwykłego `Accuracy` w późniejszej ocenie modeli.

### 2.3 Diagnostyka wycieku domenowego (`Source`, `Distance(mi)`, czas trwania)

Tabela krzyżowa (`pd.crosstab`) dla zależności `Source` vs `Severity` oraz statystyki `Distance(mi)` w klasach ciężkości.

In [ ]:
source_sev_counts = pd.crosstab(df["Source"], df[TARGET_COLUMN], margins=True, margins_name="Total")
display(source_sev_counts)

source_sev = pd.crosstab(df["Source"], df[TARGET_COLUMN], normalize="index") * 100
display(source_sev.round(2))

ax = source_sev.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="viridis")
ax.set_title("Rozkład procentowy Severity w zależności od źródła danych (Source)")
ax.set_ylabel("Udział (%)")
ax.set_xlabel("Source")
plt.legend(title="Severity", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display(df.groupby(TARGET_COLUMN, observed=True)["Distance(mi)"].agg(["median", "mean", "max"]).round(3))

**Wnioski (2.3):**
- Rozkład `Severity` silnie zależy od dostawcy danych (`Source`) — jest to artefakt sposobu kodowania w zewnętrznych API, a nie fizyczna cecha wypadku.
- `Distance(mi)` (mediana 0,473 mili w klasie `4` wobec ~0,01–0,03 w pozostałych) oraz `End_Time` opisują skutki zdarzenia po fakcie i nie są znane w chwili $t_0$.
- `Source`, `Distance(mi)` oraz `End_Time` zostają wykluczone ze zbioru cech wejściowych, aby zapobiec wyciekowi domenowemu.

### 2.4 Analiza czasowa (pokrycie kalendarza, rozkład `Severity`) i geograficzna (`State`, `City + State`, scatter plot)

Kontrola kompletności pokrycia dni w zakresie 2016–2023, normalizacja liczby zdarzeń na obserwowany dzień, procentowe rozkłady `Severity` w czasie i lokalizacjach oraz wykres rozrzutu geograficznego.

In [ ]:
start_dt_eda = pd.to_datetime(df["Start_Time"], format="mixed", errors="coerce")
eda_time = pd.DataFrame({
    "Date": start_dt_eda.dt.floor("D"),
    "Year": start_dt_eda.dt.year,
    "Weekday": start_dt_eda.dt.day_of_week,
    "Hour": start_dt_eda.dt.hour,
    "Severity": df[TARGET_COLUMN],
    "State": df["State"],
    "City": df["City"],
})

# 1. Pokrycie dni kalendarzowych i średnia dzienna liczba rekordów na obserwowany dzień
daily_counts = eda_time.groupby("Date").size()
calendar_range = pd.date_range(daily_counts.index.min(), daily_counts.index.max(), freq="D")
coverage = daily_counts.reindex(calendar_range)
print(f"Zakres dat: {calendar_range.min().date()} — {calendar_range.max().date()} | Dni bez rekordów: {int(coverage.isna().sum())}")

annual_coverage = coverage.groupby(coverage.index.year).agg(["count", "size", "sum", "mean"])
annual_coverage.columns = ["dni_z_rekordami", "dni_w_zakresie", "rekordy", "średnia_na_obserwowany_dzień"]
display(annual_coverage.round(1))

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

sns.barplot(
    x=annual_coverage.index.astype(int),
    y=annual_coverage["średnia_na_obserwowany_dzień"].to_numpy(),
    hue=annual_coverage.index.astype(int),
    palette="crest",
    dodge=False,
    ax=axes[0, 0],
)
if axes[0, 0].get_legend() is not None:
    axes[0, 0].get_legend().remove()
axes[0, 0].set_title("Średnia liczba rekordów na obserwowany dzień w roku")
axes[0, 0].set_xlabel("Rok")
axes[0, 0].set_ylabel("Rekordy / dzień z rekordami")

year_sev = pd.crosstab(eda_time["Year"], eda_time["Severity"], normalize="index") * 100
year_sev.plot(kind="bar", stacked=True, colormap="viridis", ax=axes[0, 1])
axes[0, 1].set_title("Procentowy udział klas Severity w kolejnych latach")
axes[0, 1].set_xlabel("Rok")
axes[0, 1].set_ylabel("Udział (%)")
axes[0, 1].legend(title="Severity", loc="upper right")

hour_sev = pd.crosstab(eda_time["Hour"], eda_time["Severity"], normalize="index") * 100
hour_sev.plot(kind="bar", stacked=True, colormap="viridis", legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Procentowy udział klas Severity według godziny doby")
axes[1, 0].set_xlabel("Godzina")
axes[1, 0].set_ylabel("Udział (%)")

wd_sev = pd.crosstab(eda_time["Weekday"], eda_time["Severity"], normalize="index") * 100
wd_sev.index = ["Pon", "Wt", "Śr", "Czw", "Pt", "Sob", "Nd"]
wd_sev.plot(kind="bar", stacked=True, colormap="viridis", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Procentowy udział klas Severity według dnia tygodnia")
axes[1, 1].set_xlabel("Dzień tygodnia")
axes[1, 1].set_ylabel("Udział (%)")

plt.tight_layout()
plt.show()

# 2. Mapa ciepła Weekday × Hour (znormalizowana liczbą obserwowanych dni)
hour_weekday = pd.crosstab(eda_time["Weekday"], eda_time["Hour"]).reindex(index=range(7), columns=range(24), fill_value=0)
weekday_days = daily_counts.groupby(daily_counts.index.day_of_week).size().reindex(range(7), fill_value=0)
plt.figure(figsize=(14, 4.5))
sns.heatmap(
    hour_weekday.div(weekday_days.replace(0, np.nan), axis=0),
    cmap="viridis",
    yticklabels=["Pon", "Wt", "Śr", "Czw", "Pt", "Sob", "Nd"],
)
plt.title("Średnia liczba zarejestrowanych zdarzeń w godzinie na obserwowany dzień tygodnia")
plt.xlabel("Godzina")
plt.ylabel("Dzień tygodnia")
plt.tight_layout()
plt.show()

# 3. Procentowy rozkład Severity dla top 20 stanów oraz top 20 lokalizacji (City, State)
state_counts = pd.crosstab(eda_time["State"], eda_time["Severity"])
state_top = state_counts.sum(axis=1).nlargest(20).index
state_pct = state_counts.loc[state_top].div(state_counts.loc[state_top].sum(axis=1), axis=0) * 100
state_pct.index = [f"{st} (n={state_counts.loc[st].sum():,})" for st in state_top]

city_counts = eda_time.groupby(["State", "City", "Severity"], observed=True).size().unstack("Severity", fill_value=0)
city_top = city_counts.sum(axis=1).nlargest(20).index
city_pct = city_counts.loc[city_top].div(city_counts.loc[city_top].sum(axis=1), axis=0) * 100
city_pct.index = [f"{ct}, {st} (n={city_counts.loc[(st, ct)].sum():,})" for st, ct in city_top]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
state_pct.plot.barh(stacked=True, colormap="viridis", ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_title("Udział klas Severity — 20 najliczniejszych stanów")
axes[0].set_xlabel("Udział (%)")

city_pct.plot.barh(stacked=True, colormap="viridis", legend=False, ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title("Udział klas Severity — 20 najliczniejszych lokalizacji (Miasto, Stan)")
axes[1].set_xlabel("Udział (%)")
plt.tight_layout()
plt.show()

# 4. Wykres rozrzutu geograficznego (próbka warstwowa 20 tys.)
if len(df) > 20_000:
    geo_idx, _ = train_test_split(
        df.index.to_numpy(),
        train_size=20_000,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COLUMN].to_numpy(),
    )
    geo_sample = df.loc[geo_idx, ["Start_Lng", "Start_Lat", TARGET_COLUMN]]
else:
    geo_sample = df[["Start_Lng", "Start_Lat", TARGET_COLUMN]]

fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=geo_sample,
    x="Start_Lng",
    y="Start_Lat",
    hue=TARGET_COLUMN,
    palette="viridis",
    alpha=0.45,
    s=10,
    edgecolor=None,
    ax=ax,
)
ax.set_title("Wykres rozrzutu (scatter plot): rozkład geograficzny zdarzeń wg Severity")
ax.set_xlabel("Długość geograficzna (Start_Lng)")
ax.set_ylabel("Szerokość geograficzna (Start_Lat)")
plt.tight_layout()
plt.show()

del start_dt_eda, eda_time, geo_sample
gc.collect()

**Wnioski (2.4):**
- Zakres 2016–2023 obejmuje niepełne lata brzegowe i dni bez rejestracji; rosnąca średnia dzienna liczba rekordów dotyczy zarejestrowanych zdarzeń i bez danych o ekspozycji (natężeniu ruchu) nie dowodzi wzrostu ryzyka.
- Szczyt liczby zdarzeń przypada na godziny 7–8 i 16–17 w dni robocze, lecz udział najcięższej klasy `4` rośnie w godzinach nocnych i w weekendy — uzasadnia to inżynierię cech cyklicznych (`Hour_sin/cos`, `Weekday_sin/cos`, `Month_sin/cos`) oraz flag `Is_Rush_Hour` i `Is_Weekend`.
- Udziały klas różnią się między stanami, miastami i współrzędnymi geograficznymi, a po 2020 r. widoczne jest przesunięcie rozkładu klas (dryf czasowy istotny przy projektowaniu walidacji).

### 2.5 Rozkłady cech ciągłych (histogramy i `log1p`), detekcja błędów/outlierów (boxplot, audyt IQR, violin plot), korelacja Spearmana, `pairplot` oraz pogoda i infrastruktura

Rozróżnienie błędów fizycznych czujników od naturalnych wartości odstających (boxploty i audyt nierównego wpływu filtra IQR na klasy), histogramy w skali oryginalnej i `log1p`, wykresy skrzypcowe, korelacja rang Spearmana, wykres porównawczy `pairplot` oraz struktura `Severity` według pogody, pory dnia i infrastruktury.

In [ ]:
sample_eda = df.sample(n=min(100_000, len(df)), random_state=RANDOM_STATE).copy()
weather_num_cols = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",
]

# 1. Boxploty surowych danych pogodowych + audyt IQR vs granice fizyczne
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, weather_num_cols):
    sns.boxplot(x=sample_eda[col].dropna(), color="steelblue", ax=ax)
    ax.set_title(f"Boxplot (surowe): {col}", fontsize=10)
    ax.set_xlabel("")
axes.flat[-1].axis("off")
plt.suptitle("Detekcja wartości odstających i błędów czujników przed filtracją fizyczną", y=1.02)
plt.tight_layout()
plt.show()

iqr_df = sample_eda
for col in ["Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Wind_Speed(mph)"]:
    q25, q75 = iqr_df[col].quantile([0.25, 0.75])
    margin = 1.5 * (q75 - q25)
    iqr_df = iqr_df.loc[iqr_df[col].isna() | iqr_df[col].between(q25 - margin, q75 + margin)]

iqr_audit = pd.DataFrame({
    "przed_IQR": sample_eda[TARGET_COLUMN].value_counts(),
    "po_IQR": iqr_df[TARGET_COLUMN].value_counts(),
}).fillna(0).astype(int).sort_index()
iqr_audit["usunięto_%"] = ((1 - iqr_audit["po_IQR"] / iqr_audit["przed_IQR"]) * 100).round(2)
print("Audyt wpływu filtra IQR na klasy (uzasadnienie odrzucenia IQR na rzecz granic fizycznych):")
display(iqr_audit)

sample_clean = sample_eda.copy()
error_rows = []
for col, (lo, hi) in PHYSICAL_BOUNDS.items():
    vals = pd.to_numeric(sample_clean[col], errors="coerce")
    n_err = int((vals.notna() & ~vals.between(lo, hi)).sum())
    error_rows.append({"Cecha": col, "Granice_fizyczne": f"[{lo}, {hi}]", "Błędy_w_próbce_100k": n_err})
    sample_clean[col] = vals.where(vals.between(lo, hi), np.nan)
display(pd.DataFrame(error_rows))

# 2. Histogramy cech ciągłych (skala oryginalna oraz log1p dla silnie prawoskośnych)
hist_basic = ["Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Start_Lat", "Start_Lng"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, hist_basic):
    ax.hist(sample_clean[col].dropna(), bins=30, color="#2c7fb8", edgecolor="white", linewidth=0.5)
    ax.set_title(f"Histogram: {col}", fontsize=10)
    ax.set_ylabel("Liczebność")
plt.suptitle("Histogramy cech ciągłych (po odfiltrowaniu błędów fizycznych)", y=1.02)
plt.tight_layout()
plt.show()

skewed_cols = ["Precipitation(in)", "Visibility(mi)", "Wind_Speed(mph)"]
fig, axes = plt.subplots(len(skewed_cols), 2, figsize=(13, 9))
for row_i, col in enumerate(skewed_cols):
    vals = sample_clean[col].dropna()
    axes[row_i, 0].hist(vals, bins=35, color="#2c7fb8", edgecolor="white", linewidth=0.5)
    axes[row_i, 0].set_title(f"{col} — skala oryginalna")
    axes[row_i, 1].hist(np.log1p(vals[vals >= 0]), bins=35, color="#41b6c4", edgecolor="white", linewidth=0.5)
    axes[row_i, 1].set_title(f"log1p({col}) — uwidocznienie prawego ogona")
plt.tight_layout()
plt.show()

# 3. Wykresy skrzypcowe (violin plot) kluczowych cech ciągłych w rozbiciu na klasy Severity
violin_cols = ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Wind_Speed(mph)"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, violin_cols):
    sns.violinplot(
        data=sample_clean,
        x=TARGET_COLUMN,
        y=col,
        hue=TARGET_COLUMN,
        palette="Set2",
        inner="quartile",
        dodge=False,
        ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_title(f"Violin plot: {col} wg Severity")
plt.tight_layout()
plt.show()

# 4. Korelacja rang Spearmana (heatmap) oraz zbalansowany pairplot
corr_matrix = sample_clean[weather_num_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Macierz korelacji rang Spearmana — cechy pogodowe")
plt.tight_layout()
plt.show()

pair_cols = ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Wind_Speed(mph)"]
pair_df = pd.concat([
    grp.sample(n=min(400, len(grp)), random_state=RANDOM_STATE)
    for _, grp in sample_clean[pair_cols + [TARGET_COLUMN]].groupby(TARGET_COLUMN, observed=True)
]).dropna()
g = sns.pairplot(pair_df, hue=TARGET_COLUMN, diag_kind="kde", corner=True, palette="viridis", plot_kws={"alpha": 0.4, "s": 10})
g.fig.suptitle("Wykres porównawczy (pairplot) — do 400 obserwacji na klasę", y=1.02)
plt.show()

# 5. Procentowy rozkład Severity dla Sunrise_Sunset i top 12 Weather_Condition
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, top_n in [(axes[0], "Sunrise_Sunset", 2), (axes[1], "Weather_Condition", 12)]:
    counts = pd.crosstab(df[col], df[TARGET_COLUMN])
    top_idx = counts.sum(axis=1).nlargest(top_n).index
    pct = counts.loc[top_idx].div(counts.loc[top_idx].sum(axis=1), axis=0) * 100
    pct.index = [f"{v} (n={counts.loc[v].sum():,})" for v in top_idx]
    pct.plot.barh(stacked=True, colormap="viridis", ax=ax, legend=(col == "Weather_Condition"))
    ax.invert_yaxis()
    ax.set_title(f"Rozkład Severity wg {col}")
    ax.set_xlabel("Udział (%)")
plt.tight_layout()
plt.show()

# 6. Syntetyczna tabela infrastruktury drogowej oraz redundantnych kolumn zmierzchu
infra_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
]
severe_flag = (sample_eda[TARGET_COLUMN] >= 3)
infra_rows = []
for col in infra_cols:
    mask = sample_eda[col].astype(bool)
    infra_rows.append({
        "Cecha": col,
        "Udział_True_%": round(mask.mean() * 100, 3),
        "Severe>=3_gdy_True_%": round(severe_flag[mask].mean() * 100, 2) if mask.any() else np.nan,
        "Severe>=3_gdy_False_%": round(severe_flag[~mask].mean() * 100, 2) if (~mask).any() else np.nan,
    })
display(pd.DataFrame(infra_rows).sort_values("Udział_True_%", ascending=False))

twilight_cols = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for col in twilight_cols[1:]:
    agree = (sample_eda["Sunrise_Sunset"] == sample_eda[col]).mean() * 100
    print(f"Zgodność Sunrise_Sunset vs {col}: {agree:.2f}%")

del sample_eda, sample_clean, iqr_df, pair_df
gc.collect()

**Wnioski (2.5):**
- Audyt IQR dowodzi, że mechaniczne usuwanie wartości odstających kwartylami zubaża klasy nierównomiernie (najmocniej mniejszościowe klasy `1` i `4`), usuwając rzeczywiste trudne warunki pogodowe. Właściwym podejściem są sztywne granice fizyczne (`PHYSICAL_BOUNDS`).
- Histogramy oryginalne i `log1p` potwierdzają silną prawoskośność `Precipitation(in)` i `Wind_Speed(mph)` — właściwą strategią imputacji jest **mediana**.
- Korelacja Spearmana wskazuje silną współliniowość `Temperature(F)` i `Wind_Chill(F)` ($\rho \approx 1.00$) oraz umiarkowane zależności wilgotności i opadów z widocznością ($\rho \approx -0.45$).
- Wykresy skrzypcowe i `pairplot` pokazują silne nakładanie się klas na pojedynczych cechach ciągłych. W infrastrukturze `Junction=True` wiąże się z wyższym udziałem klas ciężkich, a `Traffic_Signal`/`Crossing` z niższym; w nocy udział klasy `4` jest wyższy niż w dzień. Z 4 redundantnych kolumn zmierzchu wystarczą `Sunrise_Sunset` i `Civil_Twilight`.

## 3. Przygotowanie zbioru danych do utworzenia modelu (Etap 4)

### 3.1 Deduplikacja, selekcja kolumn wejściowych i zamrożony podział Hold-out (80/20)

Usunięcie duplikatów zdarzeń (bez `ID`), próbkowanie warstwowe, wykluczenie kolumn wyciekowych (`Source`, `Distance(mi)`, `End_Time`), stałych (`Country`, `Turning_Loop`) oraz technicznych/tekstowych, zwolnienie pełnego `df` z RAM i ścisły podział Hold-out (80/20). Wszystkie transformacje cech zostaną wykonane dopiero wewnątrz `Pipeline` (podrozdział 3.3).

In [ ]:
# Definicja grup kolumn wejściowych i pochodnych (współdzielona z własnym transformatorem i ColumnTransformerem)
BASE_NUMERIC_FEATURES = [
    "Start_Lat", "Start_Lng", "Temperature(F)", "Wind_Chill(F)",
    "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)",
    "Precipitation(in)",
]
DERIVED_NUMERIC_FEATURES = ["Temp_Chill_Diff", "Road_Features_Count"]

CATEGORICAL_FEATURES = [
    "State", "Timezone", "Weather_Condition", "Wind_Direction",
    "Sunrise_Sunset", "Civil_Twilight",
]

BINARY_FEATURES = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal",
]

CYCLICAL_AND_FLAG_FEATURES = [
    "Hour_sin", "Hour_cos", "Weekday_sin", "Weekday_cos", "Month_sin", "Month_cos",
    "Is_Weekend", "Is_Rush_Hour", "Precipitation_missing", "Wind_Chill_missing",
]

RAW_INPUT_COLUMNS = ["Start_Time"] + BASE_NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES

# 1. Odrzucenie braków celu i usunięcie duplikatów zdarzeń bez ID
valid_df = df.loc[df[TARGET_COLUMN].notna()]
dup_cols = [c for c in DUP_SUBSET if c in valid_df.columns]
dedup_mask = ~valid_df.duplicated(subset=dup_cols, keep="first")
y_full = valid_df.loc[dedup_mask, TARGET_COLUMN].astype("int8")
removed_dups = int((~dedup_mask).sum())
print(f"Usunięto zduplikowane zdarzenia (bez ID): {removed_dups:,} | Pozostało unikalnych rekordów: {len(y_full):,}")

if len(y_full) > SAMPLE_LIMIT:
    sample_idx, _ = train_test_split(
        y_full.index.to_numpy(),
        train_size=SAMPLE_LIMIT,
        random_state=RANDOM_STATE,
        stratify=y_full.to_numpy(),
    )
    work_df = df.loc[sample_idx, RAW_INPUT_COLUMNS + [TARGET_COLUMN]].copy()
else:
    work_df = df.loc[y_full.index, RAW_INPUT_COLUMNS + [TARGET_COLUMN]].copy()

del df, valid_df, dedup_mask, y_full
gc.collect()

X = work_df[RAW_INPUT_COLUMNS].copy()
y = work_df[TARGET_COLUMN].astype("int8").copy()

# 2. Ścisły podział Hold-out (80/20) przed walidacją krzyżową i jakimkolwiek dopasowaniem transformatorów
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

del work_df, X, y
gc.collect()

print(f"Zbiór treningowy (do CV): {X_train.shape[0]:,} wierszy × {X_train.shape[1]} surowych kolumn wejściowych")
print(f"Zbiór testowy (Hold-out): {X_test.shape[0]:,} wierszy × {X_test.shape[1]} surowych kolumn wejściowych")
split_summary = pd.DataFrame({
    "train_n": y_train.value_counts().sort_index(),
    "train_%": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "test_n": y_test.value_counts().sort_index(),
    "test_%": (y_test.value_counts(normalize=True).sort_index() * 100).round(2),
})
display(split_summary)

### 3.2 Implementacja walidacji krzyżowej (`StratifiedKFold`)

Podział zbioru `X_train` na 5 warstwowych foldów (`CV_FOLDS = 5`) z weryfikacją braku wspólnych indeksów między częścią treningową a walidacyjną oraz zachowania proporcji klas i współczynnika $IR$.

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv.split(X_train, y_train))

fold_rows = []
for fold_idx, (tr_idx, va_idx) in enumerate(cv_splits, start=1):
    assert len(np.intersect1d(tr_idx, va_idx)) == 0, "Wyciek: wspólne indeksy między train i val!"
    va_counts = y_train.iloc[va_idx].value_counts().sort_index()
    va_pct = (va_counts / len(va_idx) * 100).round(2)
    fold_rows.append({
        "Fold": fold_idx,
        "Train_n": len(tr_idx),
        "Val_n": len(va_idx),
        "Val_klasa_1_%": va_pct.get(1, 0.0),
        "Val_klasa_2_%": va_pct.get(2, 0.0),
        "Val_klasa_3_%": va_pct.get(3, 0.0),
        "Val_klasa_4_%": va_pct.get(4, 0.0),
        "Val_IR_max": round(va_counts.max() / va_counts.min(), 2),
    })

display(pd.DataFrame(fold_rows))

### 3.3 Własny transformator (`BaseEstimator`, `TransformerMixin`), `Pipeline`, balansowanie klas oraz konfiguracja `GridSearchCV`

Zgodnie z wytycznymi *scikit-learn developers*:
- **`AccidentFeatureEngineer(BaseEstimator, TransformerMixin)`** zamyka całą inżynierię cech (granice fizyczne, ujednolicenie `Wind_Direction`, flagi braków, kodowanie cykliczne czasu `sin`/`cos`, flagi `Is_Weekend`/`Is_Rush_Hour`, `Temp_Chill_Diff`, `Road_Features_Count`) w klasie zgodnej z `clone()`, `get_params()`, `set_params()` i bez mutowania stanu podczas `transform()`.
- **`build_full_pipeline()`** składa kompletny `ImbPipeline`: `fe` → `prep` (`ColumnTransformer` z dynamicznymi selektorami kolumn) → `var` (`VarianceThreshold`) → `bal` (resampling wewnątrz foldu) → `clf` (placeholder pod model z Etapu 5).
- **`build_grid_search()`** przygotowuje obiekt `GridSearchCV` z `StratifiedKFold`, 4 metrykami dla klas niezbalansowanych (`f1_macro`, `mcc`, `bal_acc`, `recall_c4`) oraz gotową siatką `PREPROCESSING_PARAM_GRID`, umożliwiającą równoczesne strojenie inżynierii cech, preprocessingu, balansowania i hiperparametrów modelu w Etapach 5–8.

In [ ]:
class AccidentFeatureEngineer(BaseEstimator, TransformerMixin):
    """Transformator inżynierii cech zgodny z kontraktem scikit-learn (BaseEstimator, TransformerMixin)."""

    def __init__(
        self,
        clip_physical_bounds=True,
        normalize_wind=True,
        add_missing_flags=True,
        add_cyclical_time=True,
        add_domain_features=True,
    ):
        self.clip_physical_bounds = clip_physical_bounds
        self.normalize_wind = normalize_wind
        self.add_missing_flags = add_missing_flags
        self.add_cyclical_time = add_cyclical_time
        self.add_domain_features = add_domain_features

    def _compute_output_columns(self, input_cols: list[str]) -> list[str]:
        cols = list(input_cols)
        if self.add_missing_flags:
            if "Precipitation(in)" in cols:
                cols.append("Precipitation_missing")
            if "Wind_Chill(F)" in cols:
                cols.append("Wind_Chill_missing")
        if "Start_Time" in cols:
            if self.add_cyclical_time:
                cols.extend(["Hour_sin", "Hour_cos", "Weekday_sin", "Weekday_cos", "Month_sin", "Month_cos"])
            if self.add_domain_features:
                cols.extend(["Is_Weekend", "Is_Rush_Hour"])
            cols.remove("Start_Time")
        if self.add_domain_features:
            if "Temperature(F)" in cols and "Wind_Chill(F)" in cols:
                cols.append("Temp_Chill_Diff")
            if any(c in cols for c in BINARY_FEATURES):
                cols.append("Road_Features_Count")
        return cols

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("AccidentFeatureEngineer wymaga obiektu pandas.DataFrame.")
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        self.output_features_ = np.asarray(self._compute_output_columns(list(X.columns)), dtype=object)
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, "is_fitted_")
        if not isinstance(X, pd.DataFrame):
            raise TypeError("AccidentFeatureEngineer wymaga obiektu pandas.DataFrame.")

        out = X.copy()

        # 1. Sztywne granice fizyczne czujników pogodowych
        if self.clip_physical_bounds:
            for col, (lo, hi) in PHYSICAL_BOUNDS.items():
                if col in out.columns:
                    vals = pd.to_numeric(out[col], errors="coerce")
                    out[col] = vals.where(vals.between(lo, hi), np.nan).astype("float32")
        else:
            for col in PHYSICAL_BOUNDS:
                if col in out.columns:
                    out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")

        # 2. Ujednolicenie synonimów w Wind_Direction
        if "Wind_Direction" in out.columns:
            wd = out["Wind_Direction"].astype("object")
            wd = wd.where(wd.notna(), np.nan)
            if self.normalize_wind:
                wd = wd.replace(WIND_ALIASES)
            out["Wind_Direction"] = wd

        # 3. Flagi nielosowych braków pogodowych
        if self.add_missing_flags:
            if "Precipitation(in)" in out.columns:
                out["Precipitation_missing"] = out["Precipitation(in)"].isna().astype("int8")
            if "Wind_Chill(F)" in out.columns:
                out["Wind_Chill_missing"] = out["Wind_Chill(F)"].isna().astype("int8")

        # 4. Cechy czasowe ze Start_Time
        if "Start_Time" in out.columns:
            start_dt = pd.to_datetime(out["Start_Time"], format="mixed", errors="coerce")
            hour = start_dt.dt.hour
            weekday = start_dt.dt.day_of_week
            month = start_dt.dt.month

            if self.add_cyclical_time:
                out["Hour_sin"] = np.sin(2 * np.pi * hour / 24.0).astype("float32")
                out["Hour_cos"] = np.cos(2 * np.pi * hour / 24.0).astype("float32")
                out["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0).astype("float32")
                out["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0).astype("float32")
                out["Month_sin"] = np.sin(2 * np.pi * (month - 1) / 12.0).astype("float32")
                out["Month_cos"] = np.cos(2 * np.pi * (month - 1) / 12.0).astype("float32")

            if self.add_domain_features:
                out["Is_Weekend"] = weekday.isin([5, 6]).astype("int8")
                out["Is_Rush_Hour"] = (hour.between(6, 9) | hour.between(15, 18)).astype("int8")

            out = out.drop(columns=["Start_Time"])

        # 5. Konwersja flag binarnych na 0/1
        present_binary = [c for c in BINARY_FEATURES if c in out.columns]
        for col in present_binary:
            out[col] = out[col].fillna(False).astype(bool).astype("int8")

        # 6. Cechy domenowe pogodowe i infrastrukturalne
        if self.add_domain_features:
            if "Temperature(F)" in out.columns and "Wind_Chill(F)" in out.columns:
                out["Temp_Chill_Diff"] = (out["Temperature(F)"] - out["Wind_Chill(F)"]).astype("float32")
            if present_binary:
                out["Road_Features_Count"] = out[present_binary].sum(axis=1).astype("float32")

        for col in CATEGORICAL_FEATURES:
            if col in out.columns and col != "Wind_Direction":
                out[col] = out[col].astype("object").where(out[col].notna(), np.nan)

        return out

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "output_features_")
        return self.output_features_


# Dynamiczne selektory kolumn dla ColumnTransformer (dostosowują się do włączonych flag w AccidentFeatureEngineer)
def select_numeric_columns(X):
    allowed = set(BASE_NUMERIC_FEATURES + DERIVED_NUMERIC_FEATURES)
    return [c for c in X.columns if c in allowed]


def select_categorical_columns(X):
    allowed = set(CATEGORICAL_FEATURES)
    return [c for c in X.columns if c in allowed]


def select_passthrough_columns(X):
    allowed = set(CYCLICAL_AND_FLAG_FEATURES + BINARY_FEATURES)
    return [c for c in X.columns if c in allowed]


# Strategie rozwiązania niezbalansowania klas
def soft_rus_strategy(y_arr):
    counts = pd.Series(y_arr).value_counts().sort_values(ascending=False)
    if len(counts) < 2:
        return counts.to_dict()
    cap = int(counts.iloc[1] * 2)
    return {cls: min(int(cnt), cap) for cls, cnt in counts.items()}


def smote_after_rus_strategy(y_arr):
    counts = pd.Series(y_arr).value_counts()
    target = int(counts.median())
    return {cls: max(int(cnt), target) for cls, cnt in counts.items()}


def build_balancer_steps(strategy: str) -> list:
    if strategy in ("none", "class_weight"):
        return [("bal", "passthrough")]
    if strategy == "rus_full":
        return [("bal", RandomUnderSampler(sampling_strategy="auto", random_state=RANDOM_STATE))]
    if strategy == "rus_soft":
        return [("bal", RandomUnderSampler(sampling_strategy=soft_rus_strategy, random_state=RANDOM_STATE))]
    if strategy == "smote_rus":
        return [
            ("rus", RandomUnderSampler(sampling_strategy=soft_rus_strategy, random_state=RANDOM_STATE)),
            ("smote", SMOTE(sampling_strategy=smote_after_rus_strategy, k_neighbors=3, random_state=RANDOM_STATE)),
        ]
    raise ValueError(f"Nieznana strategia balansowania: {strategy}")


def build_preprocessing_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), select_numeric_columns),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=0.01,
                    sparse_output=False,
                    dtype=np.float32,
                )),
            ]), select_categorical_columns),
            ("pass", SimpleImputer(strategy="constant", fill_value=0), select_passthrough_columns),
        ],
        remainder="drop",
    )
    return Pipeline(steps=[
        ("fe", AccidentFeatureEngineer()),
        ("prep", preprocessor),
        ("var", VarianceThreshold(threshold=0.0)),
    ])


def build_full_pipeline(classifier="passthrough", balancer: str = "class_weight") -> ImbPipeline:
    """Kompletny rurociąg gotowy do wpięcia dowolnego klasyfikatora w Etapie 5."""
    prep_steps = build_preprocessing_pipeline().steps
    return ImbPipeline(steps=prep_steps + build_balancer_steps(balancer) + [("clf", classifier)])


# Konfiguracja metryk i fabryka GridSearchCV (gotowa do uruchomienia w Etapie 5)
def recall_class_4(y_true, y_pred):
    return recall_score(y_true, y_pred, labels=[4], average=None, zero_division=0)[0]


SCORING_METRICS = {
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
    "mcc": make_scorer(matthews_corrcoef),
    "bal_acc": make_scorer(balanced_accuracy_score),
    "recall_c4": make_scorer(recall_class_4),
}

PREPROCESSING_PARAM_GRID = {
    "fe__add_domain_features": [True, False],
    "fe__add_missing_flags": [True, False],
    "prep__num__imputer__strategy": ["median", "mean"],
    "prep__cat__ohe__min_frequency": [0.005, 0.01, 0.02],
}


def build_grid_search(
    classifier,
    param_grid: dict | list[dict],
    balancer: str = "class_weight",
    refit_metric: str = "f1_macro",
) -> GridSearchCV:
    """Tworzy gotowy obiekt GridSearchCV z bezwyciekowym ImbPipeline i warstwową walidacją CV."""
    return GridSearchCV(
        estimator=build_full_pipeline(classifier=classifier, balancer=balancer),
        param_grid=param_grid,
        cv=cv,
        scoring=SCORING_METRICS,
        refit=refit_metric,
        n_jobs=N_JOBS,
        return_train_score=False,
    )


# Wagi klas na zbiorze treningowym (informacyjnie — w CV liczone osobno w każdym foldzie)
classes = np.sort(y_train.unique())
class_weights_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.to_numpy())
class_weights_dict = {int(c): round(float(w), 4) for c, w in zip(classes, class_weights_arr)}
sample_weights_train = pd.Series(
    compute_sample_weight(class_weight="balanced", y=y_train),
    index=y_train.index,
    name="sample_weight",
)
print("Wagi klas (class_weight='balanced'):", class_weights_dict)
display(sample_weights_train.describe().to_frame().round(4))

### 3.4 Weryfikacja kontraktu `Pipeline` / `GridSearchCV` (dry-run na Foldzie 0, bez trenowania modelu) i zapis bazy

Kontrolny test bez żadnego klasyfikatora:
1. Weryfikacja zgodności `AccidentFeatureEngineer` z API scikit-learn (`clone`, `set_params`, dopasowanie wyłącznie na podzbiorze treningowym Folda 0, 0 wartości `NaN` w treningu i walidacji).
2. Weryfikacja poprawności wszystkich kluczy siatki `PREPROCESSING_PARAM_GRID` w konfiguracji `GridSearchCV`.
3. Pomiar redukcji współczynnika niezbalansowania $IR_{\max}$ dla strategii resamplingu na części treningowej Folda 0 (przy nietkniętej walidacji).
4. Zapis czystych podziałów danych i konfiguracji do pliku `prepared_data_stage4.joblib` pod Etap 5.

In [ ]:
tr0_idx, va0_idx = cv_splits[0]
X_tr0, y_tr0 = X_train.iloc[tr0_idx], y_train.iloc[tr0_idx]
X_va0, y_va0 = X_train.iloc[va0_idx], y_train.iloc[va0_idx]

# 1. Test pełnego Pipeline (fit WYŁĄCZNIE na części treningowej Folda 0)
prep_pipe_full = build_preprocessing_pipeline()
X_tr0_proc = prep_pipe_full.fit_transform(X_tr0)
X_va0_proc = prep_pipe_full.transform(X_va0)

raw_feat_names = prep_pipe_full.named_steps["prep"].get_feature_names_out()
final_feat_names = raw_feat_names[prep_pipe_full.named_steps["var"].get_support()]

# 2. Test klonowania i modyfikacji parametrów własnego transformatora (wymóg GridSearchCV)
prep_pipe_alt = clone(prep_pipe_full)
prep_pipe_alt.set_params(
    fe__add_domain_features=False,
    fe__add_missing_flags=False,
    prep__cat__ohe__min_frequency=0.02,
)
X_tr0_alt = prep_pipe_alt.fit_transform(X_tr0)

print(f"Fold 0 (pełny FE)   -> train: {X_tr0_proc.shape}, val: {X_va0_proc.shape} | NaN: {np.isnan(X_tr0_proc).sum() + np.isnan(X_va0_proc).sum()}")
print(f"Fold 0 (okrojony FE)-> train: {X_tr0_alt.shape} | NaN: {np.isnan(X_tr0_alt).sum()}")
print(f"Liczba cech wyjściowych po One-Hot i VarianceThreshold: {len(final_feat_names)}")

# 3. Walidacja konfiguracji GridSearchCV bez trenowania klasyfikatora
grid_template = build_grid_search(classifier="passthrough", param_grid=PREPROCESSING_PARAM_GRID, balancer="class_weight")
pipeline_params = set(grid_template.estimator.get_params().keys())
missing_keys = [k for k in PREPROCESSING_PARAM_GRID if k not in pipeline_params]
assert not missing_keys, f"Niepoprawne klucze w param_grid: {missing_keys}"
print(f"Walidacja GridSearchCV OK — wszystkie {len(PREPROCESSING_PARAM_GRID)} klucze siatki istnieją w Pipeline.")

# 4. Wpływ strategii balansowania wyłącznie na podzbiór treningowy Folda 0
balance_rows = []
for strat in ["none", "rus_soft", "smote_rus", "rus_full"]:
    if strat == "none":
        y_res = y_tr0.to_numpy()
    else:
        resampler = ImbPipeline(steps=build_balancer_steps(strat))
        _, y_res = resampler.fit_resample(X_tr0_proc, y_tr0.to_numpy())
    cnts = pd.Series(y_res).value_counts().sort_index()
    balance_rows.append({
        "Strategia": strat,
        "Train_0_n": int(len(y_res)),
        "Klasa_1": int(cnts.get(1, 0)),
        "Klasa_2": int(cnts.get(2, 0)),
        "Klasa_3": int(cnts.get(3, 0)),
        "Klasa_4": int(cnts.get(4, 0)),
        "IR_max": round(cnts.max() / cnts.min(), 2),
    })

print("\nWpływ strategii balansowania na zbiór treningowy Folda 0 (walidacja i Hold-out nietknięte):")
display(pd.DataFrame(balance_rows))

# 5. Zapis czystej bazy danych (wyłącznie standardowe typy Pythona/Pandas -> bezproblemowy odczyt w Etapie 5)
prepared_bundle = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "cv_splits": cv_splits,
    "raw_input_columns": RAW_INPUT_COLUMNS,
    "class_weights": class_weights_dict,
    "preprocessing_param_grid": PREPROCESSING_PARAM_GRID,
}
joblib.dump(prepared_bundle, PREPARED_ARTIFACT_PATH, compress=3)
print(f"\nZapisano gotową bazę do dalszego modelowania: {PREPARED_ARTIFACT_PATH}")

**Wniosek końcowy (Etap 4):**
- Cały proces czyszczenia bezstanowego i inżynierii cech został zamknięty w niestandardowym transformatorze `AccidentFeatureEngineer(BaseEstimator, TransformerMixin)` zgodnym ze standardem *scikit-learn* i podłączony do `ImbPipeline` oraz fabryki `GridSearchCV`.
- Dzięki dynamicznym selektorom kolumn w `ColumnTransformer` wszystkie parametry inżynierii cech (`fe__*`) oraz kodowania i imputacji (`prep__*`) mogą być bezpośrednio strojone razem z hiperparametrami modeli w Etapach 5–8 bez ryzyka wycieku danych.
- Test kontrolny na Foldzie 0 potwierdził 0 braków po transformacji, poprawność klonowania/parametryzacji oraz redukcję $IR_{\max}$ wyłącznie w obrębie podzbioru treningowego. Gotowe podziały danych zapisano w `prepared_data_stage4.joblib`.